In [2]:
import tensorflow as tf

from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(X_train.shape, X_valid.shape, X_test.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(55000, 28, 28) (5000, 28, 28) (10000, 28, 28)


In [3]:
def image_to_tfexample(image_data, label):
    feature = {
        'image': Feature(bytes_list=BytesList(value=[image_data.tobytes()])),
        'label': Feature(int64_list=Int64List(value=[label]))
    }
    return Example(features=Features(feature=feature))

def write_tfrecords(filename, images, labels):
    with tf.io.TFRecordWriter(filename) as f:
        for image, label in zip(images, labels):
            example = image_to_tfexample(image, label)
            f.write(example.SerializeToString())

print(image_to_tfexample(X_train[0], y_train[0]))
write_tfrecords("fashion_mist_train.tfrecords", X_train, y_train)
write_tfrecords("fashion_mist_valid.tfrecords", X_valid, y_valid)
write_tfrecords("fashion_mist_test.tfrecords", X_test, y_test)

features {
  feature {
    key: "label"
    value {
      int64_list {
        value: 9
      }
    }
  }
  feature {
    key: "image"
    value {
      bytes_list {
        value: "\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\001\000\000\rI\000\000\001\004\000\000\000\000\001\001\000\000\000\000\000\000\000\000\000\000\000\000\000\003\000$\210\177>6\000\000\000\001\003\004\000\000\003\000\000\000\000\000\000\000\000\000\000\000\000\006\000f\314\260\206\220{\027\000\000\000\000\014\n\000\000\000\000\000\000\000\000\000\000\000\000\000\000\000\233\354\317\262k\234\241m@\027M\202H\017\000\000\000\000\000\000\000\000\000\000\000\001\000E\317\337\332\330\3

In [4]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(32)
valid_ds = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(32)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

print(train_ds)


<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28), dtype=tf.uint8, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>


In [11]:
from tensorflow.keras.callbacks import EarlyStopping
import os

tf.random.set_seed(42)
norm_layer = tf.keras.layers.Normalization()
norm_layer.adapt(train_ds.map(lambda x, y: x))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    # tf.keras.layers.Rescaling(1./255),
    norm_layer,
    tf.keras.layers.Flatten(),

    # norm_layer,
    tf.keras.layers.Dense(300, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',   # Metric to monitor
    patience=5,           # How many epochs to wait for improvement
    restore_best_weights=True # Keep the weights from the best epoch
)

root_logdir = os.path.join(os.curdir, "my_logs")

def get_run_logdir():
    import time
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()

tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)


history = model.fit(train_ds, epochs=30, validation_data=valid_ds, callbacks=[early_stopping, tensorboard_cb])

%load_ext tensorboard
%tensorboard --logdir=./my_logs --port=6006

Epoch 1/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8121 - loss: 0.5327 - val_accuracy: 0.8526 - val_loss: 0.3999
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8649 - loss: 0.3770 - val_accuracy: 0.8690 - val_loss: 0.3584
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8779 - loss: 0.3365 - val_accuracy: 0.8754 - val_loss: 0.3391
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8872 - loss: 0.3101 - val_accuracy: 0.8802 - val_loss: 0.3276
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8954 - loss: 0.2895 - val_accuracy: 0.8832 - val_loss: 0.3208
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9017 - loss: 0.2722 - val_accuracy: 0.8852 - val_loss: 0.3167
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9075 - loss: 0.2573 - val_accuracy: 0.8862 - val_loss: 0.3135
Epoch 8/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9126 - loss: 0.2437 - 